### News Sentiment (NewsAPI + Finnhub, last 10 days)

- **Output**: `Reports/news_cleaned_df.csv` (relevant non-neutral articles, used by the app's AI news summaries),
  `Reports/weighted_sentiment.csv` (SentimentScore −10..+10 for **every** tracked symbol; 0 = no relevant news),
  `Reports/sentiment_history.csv` (appended each online run: date, symbol, score, article counts).
- **Modes** (env `PIPELINE_SENTIMENT_MODE`, default `online` when run by hand; `run_all.py` sets it):
  - `online` — fetch news for all symbols (≈97 NewsAPI + 97 Finnhub calls (96 stocks + QQQ; NewsAPI free limit 100/day)), score with FinBERT, write outputs.
  - `offline` — no network: re-apply the relevance filter to the cached `news_cleaned_df.csv` and recompute the scores
    (idempotent; used by `run_all.py` in quick mode and whenever NewsAPI already ran in the last 24 h).
- **Quota guard**: run by hand in `online` mode, the notebook refuses to call NewsAPI again within 24 h of the last run
  (read from `Reports/run_state.json`); set `PIPELINE_FORCE_NEWS=1` to override.
  Every NewsAPI/Finnhub call is counted and printed at the end (and reported in the `run_all.py` summary).
  - `sample` — fetch 1–2 symbols (`PIPELINE_SAMPLE_SYMBOLS`, default `NVDA`) end to end and write only to `Reports/cache/`.
- Relevance: `is_relevant` in the **Relevance filter** section (ticker forms or distinctive company names/aliases).

### Setup
Mode (online / offline / sample), symbols, API keys from `.env` (never printed) and file paths.

In [1]:
import json
import os
import time
from datetime import datetime, timedelta, timezone

import pandas as pd
from dotenv import load_dotenv

import sector_mapping
from sector_mapping import stock_symbols

load_dotenv(os.path.join(sector_mapping.PROJECT_ROOT, ".env"))
MODE = os.getenv("PIPELINE_SENTIMENT_MODE", "online").lower()
assert MODE in {"online", "offline", "sample"}, MODE
REPORTS_DIR = sector_mapping.REPORTS_DIR
CACHE_DIR = os.path.join(REPORTS_DIR, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)
NEWS_CSV = os.path.join(REPORTS_DIR, "news_cleaned_df.csv")
SCORE_CSV = os.path.join(REPORTS_DIR, "weighted_sentiment.csv")
HISTORY_CSV = os.path.join(REPORTS_DIR, "sentiment_history.csv")
if MODE == "sample":
    SYMBOLS = [s.strip().upper() for s in os.getenv("PIPELINE_SAMPLE_SYMBOLS", "NVDA").split(",")][:2]
else:
    SYMBOLS = stock_symbols

END = datetime.now(timezone.utc).date()
START = END - timedelta(days=10)
# Asymmetric FinBERT confidence cutoffs: positive needs >= 0.75, negative >= 0.65
# (negative financial language is subtler, so its bar is lower). See "Filter & score".
POSITIVE_CUTOFF, NEGATIVE_CUTOFF = 0.75, 0.65
PRIOR_ARTICLES = 5
print(f"Mode: {MODE} | symbols: {len(SYMBOLS)} | window {START} → {END}")


def with_retries(fn, tries=3, base_wait=2):
    """Call fn(); retry transient failures with exponential backoff, re-raise the last error."""
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            if attempt == tries - 1 or "rateLimited" in str(e) or "maximumResultsReached" in str(e):
                raise
            time.sleep(base_wait * 2 ** attempt)

Mode: online | symbols: 97 | window 2026-09-16 → 2026-09-26


### Relevance filter (is this article really about the stock?)
An article counts for a symbol when ANY of these holds: an explicit ticker form (`$TICK`, `(TICK)`, `NASDAQ:TICK`,
`TICK stock/shares`); the bare upper-case ticker, unless it is also a common word (`AMBIGUOUS_TICKERS`); or a distinctive
company name / brand alias as whole words (`ALIASES`; names in `CASE_SENSITIVE_ALIASES` must match the case; a tuple means
all its words must appear). The old rule matched the first word of the company name, so "advanced" (AMD) or "super" (SMCI)
matched unrelated news. **When you add a stock to the universe, add its aliases here.**

In [2]:
import re

from sector_mapping import symbol_name

SUFFIXES = {"inc", "inc.", "corp", "corp.", "corporation", "co", "co.", ".inc", "ltd", "ltd.",
            "plc", "group", "holding", "holdings", "company", "companies", "nv", "n.v."}

AMBIGUOUS_TICKERS = {"U", "UI", "IT", "APP", "NOW", "ARM", "TEAM", "HOOD", "FIG", "CRM", "COIN", "SNOW", "SHOP",
                     "DASH", "MU", "ZS", "TEM", "HAL", "UBER", "META", "MARA", "HIMS", "SOFI", "A",
                     "APA", "FANG", "URI",   # 2026-09-24 U91 additions: APA (psych. assoc.), FANG (= FAANG stocks), URI (web term)
                     "CLS", "LITE"}          # 2026-09-24 U96 additions: CLS (CLS Bank, Cumulative Layout Shift), LITE (the word "lite")

ALIASES = {
    "AAPL": ["Apple"], "ADBE": ["Adobe"], "AFRM": ["Affirm Holdings", "Affirm"], "AMZN": ["Amazon"],
    "ANET": ["Arista Networks", "Arista"], "AMD": ["Advanced Micro Devices"], "APP": ["AppLovin"],
    "AVAV": ["AeroVironment"], "AVGO": ["Broadcom"], "BIIB": ["Biogen"], "BKR": ["Baker Hughes"],
    "CDNS": ["Cadence Design"], "COIN": ["Coinbase"], "CRM": ["Salesforce"], "CRWV": ["CoreWeave"],
    "CVLT": ["Commvault"], "DASH": ["DoorDash"], "ENPH": ["Enphase"], "FIG": ["Figma"], "FTNT": ["Fortinet"],
    "GOOGL": ["Alphabet", "Google"], "GTLB": ["GitLab"], "HAL": ["Halliburton"],
    "HIMS": ["Hims & Hers", "Hims and Hers", "Hims&Hers"], "HOOD": ["Robinhood"], "INTU": ["Intuit", "TurboTax"],
    "IREN": ["Iris Energy", "IREN Limited"], "LRCX": ["Lam Research"], "MCK": ["McKesson"], "MRK": ["Merck"],
    "META": ["Meta Platforms", "Meta", "Facebook"], "MRVL": ["Marvell"], "MSFT": ["Microsoft"], "MU": ["Micron"],
    "NFLX": ["Netflix"], "NOW": ["ServiceNow"], "NVDA": ["Nvidia"], "ORCL": ["Oracle"], "PANW": ["Palo Alto Networks"],
    "RCL": ["Royal Caribbean"], "REGN": ["Regeneron"], "SLB": ["Schlumberger"], "SNOW": ["Snowflake"],
    "SOFI": ["SoFi"], "TEAM": ["Atlassian"], "TSLA": ["Tesla"], "UBER": ["Uber"], "UI": ["Ubiquiti"],
    "UNH": ["UnitedHealth"], "UPST": ["Upstart"], "VEEV": ["Veeva"], "VRT": ["Vertiv"], "ZS": ["Zscaler"],
    "UMAC": ["Unusual Machines"], "NOC": ["Northrop Grumman", "Northrop"], "MDB": ["MongoDB"],
    "LMT": ["Lockheed Martin", "Lockheed"], "U": ["Unity Software", "Unity Technologies", ("Unity", "stock")], "TWLO": ["Twilio"],
    "CRCL": ["Circle Internet", ("Circle", "USDC"), ("Circle", "stablecoin"), ("Circle", "stablecoins")], "ACHR": ["Archer Aviation"],
    "ALAB": ["Astera Labs"], "APLD": ["Applied Digital"], "ARM": ["Arm Holdings", "Arm"], "ASTS": ["AST SpaceMobile"],
    "CIFR": ["Cipher Mining", "Cipher Digital"], "CVNA": ["Carvana"], "IONQ": ["IonQ"], "JOBY": ["Joby Aviation", "Joby"],
    "MARA": ["MARA Holdings", "Marathon Digital"], "MSTR": ["MicroStrategy", "Strategy Inc", "Michael Saylor", "Saylor"],
    "NVTS": ["Navitas Semiconductor", "Navitas"], "RDDT": ["Reddit"], "RKLB": ["Rocket Lab"], "SHOP": ["Shopify"],
    "SMCI": ["Super Micro", "Supermicro"], "SOUN": ["SoundHound"], "TEM": ["Tempus AI"], "QQQ": ["Invesco QQQ"],
    # U91 additions (2026-09-24)
    "APA": ["APA Corporation", "APA Corp", "Apache Corporation"], "OXY": ["Occidental Petroleum", "Occidental"],
    "TRGP": ["Targa Resources", "Targa"], "DVN": ["Devon Energy"], "FANG": ["Diamondback Energy"],
    "COF": ["Capital One"], "C": ["Citigroup", "Citibank", ("Citi", "bank")], "BE": ["Bloom Energy"], "BA": ["Boeing"],
    "URI": ["United Rentals"], "PH": ["Parker-Hannifin", "Parker Hannifin"], "FCX": ["Freeport-McMoRan", "Freeport McMoRan"],
    "LYB": ["LyondellBasell"],
    # U96 additions (2026-09-24)
    "CRDO": ["Credo Technology", ("Credo", "semiconductor"), ("Credo", "connectivity"), ("Credo", "AI")], "NBIS": ["Nebius"],
    "LITE": ["Lumentum"], "CLS": ["Celestica"], "RBRK": ["Rubrik"],
}
CASE_SENSITIVE_ALIASES = {"Meta", "Uber", "Affirm", "Joby", "Navitas", "Oracle", "Apple", "Amazon", "Tempus AI",
                          "Arm Holdings", "Arm", "Circle", "Unity", "Merck", "Northrop", "Lockheed", "Intuit",
                          "Occidental", "Targa", "Boeing", "Citi", "Credo"}


def clean_company_name(name):
    """Company name without punctuation and legal suffixes (Inc, Corp, ...)."""
    name = re.sub(r"\.com\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"[^\w\s&]", "", name)
    return " ".join(w for w in name.split() if w.lower() not in SUFFIXES).strip()


def aliases_for(symbol):
    """Curated aliases first, then the full cleaned company name (never a single generic first word)."""
    out = list(ALIASES.get(symbol, []))
    full = clean_company_name(symbol_name.get(symbol, ""))
    if full and full not in out and len(full.split()) >= 2:
        out.append(full)
    return out


def search_query(symbol):
    """NewsAPI query: the most distinctive alias as an exact phrase, OR the ticker when it is not a common word."""
    names = [a for a in aliases_for(symbol) if isinstance(a, str)]
    q = f'"{names[0]}"' if names else symbol
    if names and symbol not in AMBIGUOUS_TICKERS and len(symbol) >= 3:
        q += f" OR {symbol}"
    return q


def _has_phrase(phrase, text, case_sensitive):
    """True when the phrase appears as a whole word in the text."""
    flags = 0 if case_sensitive else re.IGNORECASE
    return re.search(r"(?<![\w$])" + re.escape(phrase) + r"(?!\w)", text, flags) is not None


def is_relevant(symbol, text):
    """True when the article text clearly refers to `symbol`."""
    if not isinstance(text, str) or not text:
        return False
    t = re.escape(symbol)
    strong = (rf"\${t}\b", rf"\({t}\)", rf"\b(?:NASDAQ|NYSE|Nasdaq|NYSEARCA|AMEX)\s*:\s*{t}\b", rf"\b{t}\s+(?:stock|shares)\b")
    if any(re.search(p, text) for p in strong):
        return True
    if symbol not in AMBIGUOUS_TICKERS and len(symbol) >= 3 and re.search(rf"(?<![\w$]){t}(?!\w)", text):
        return True
    for alias in aliases_for(symbol):
        terms = alias if isinstance(alias, tuple) else (alias,)
        if all(_has_phrase(a, text, a in CASE_SENSITIVE_ALIASES) for a in terms):
            return True
    return False

### Fetch news (online / sample only)

Two sources per symbol over the last 10 days: NewsAPI (`get_everything`, 25 articles/symbol) and Finnhub `company_news`.

- **Quota guards:** NewsAPI's free tier is 100 calls/day and one run uses ~97, so the notebook shares `run_state.json`
  with `run_all.py` and refuses to call NewsAPI twice within 24 h (override with `PIPELINE_FORCE_NEWS=1`).
  Finnhub's free tier is 60 calls/min — hence the 1.1 s sleep after every call, including failed ones.
- **Resilience:** every call goes through `with_retries` (3 attempts, exponential backoff) and per-symbol failures
  are collected in `errors` instead of aborting the run. If NewsAPI rate-limits, the loop breaks early and continues
  with Finnhub only. If *zero* articles come back in online mode, the notebook raises rather than wiping the outputs.

In [3]:
articles, errors = [], []
CALLS = {"newsapi": 0, "finnhub": 0}


def counted(api, fn):
    """Count every request attempt (retries included) for the run summary."""
    def call():
        CALLS[api] += 1
        return fn()
    return call


STATE_FILE = os.path.join(REPORTS_DIR, "run_state.json")   # shared with run_all.py: NewsAPI at most once per 24 h
NEWSAPI_DAILY_CAP = 98          # NewsAPI free tier: 100 calls/day; stop issuing new calls at 98 attempts (retries included)
STAMP_NEWS = MODE == "online" and os.getenv("PIPELINE_NEWS_APPROVED") != "1"   # run by hand (run_all.py applies the guard itself)
if STAMP_NEWS:
    state = json.load(open(STATE_FILE)) if os.path.exists(STATE_FILE) else {}
    last = state.get("last_news_at")
    if last and os.getenv("PIPELINE_FORCE_NEWS") != "1":
        gap_h = (datetime.now(timezone.utc) - datetime.fromisoformat(last)).total_seconds() / 3600
        if gap_h < 24:
            raise RuntimeError(f"NewsAPI already ran {gap_h:.1f} h ago (free limit 100 calls/day, one run = 97). "
                               "Set PIPELINE_FORCE_NEWS=1 to override, or use PIPELINE_SENTIMENT_MODE=offline.")
    # NOTE: last_news_at is stamped AFTER the NewsAPI loop below, not here - a pre-call crash
    # used to create a false 24 h lockout that skipped the next day's update.

if MODE in {"online", "sample"}:
    import finnhub
    from newsapi import NewsApiClient

    newsapi = NewsApiClient(api_key=os.getenv("NEWS_API_KEY"))
    finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))
    NEWS_RAW_CSV = os.path.join(CACHE_DIR, f"news_raw_{END.isoformat()}.csv")   # today's raw NewsAPI payload
    try:
        raw_df = pd.read_csv(NEWS_RAW_CSV)
        raw_done = set(raw_df["symbol"].str.upper())
    except Exception:
        raw_df, raw_done = pd.DataFrame(), set()   # no raw file yet (or unreadable) - fetch everything
    for symbol in SYMBOLS:
        if symbol in raw_done:
            # idempotent retry: reuse today's already-fetched payload instead of re-spending quota
            prev = raw_df[raw_df["symbol"].str.upper() == symbol]
            articles += prev.to_dict("records")
            print(f"  {symbol}: already fetched today - reusing {len(prev)} raw articles")
            continue
        if CALLS["newsapi"] >= NEWSAPI_DAILY_CAP:   # circuit breaker: retries count too, stay under 100/day
            print(f"NewsAPI attempt cap ({NEWSAPI_DAILY_CAP}/day) reached - continuing with Finnhub only")
            errors.append(("newsapi", symbol, f"stopped at daily attempt cap {NEWSAPI_DAILY_CAP}"))
            break
        try:
            res = with_retries(counted("newsapi", lambda: newsapi.get_everything(
                q=search_query(symbol), language="en", from_param=START.isoformat(), to=END.isoformat(), sort_by="publishedAt",
                page_size=25)))
        except Exception as e:
            errors.append(("newsapi", symbol, str(e)[:120]))
            if "rateLimited" in str(e):
                print("NewsAPI daily limit reached - continuing with Finnhub only")
                break
            continue
        new_arts = [{"symbol": symbol, "date": a.get("publishedAt"), "headline": a.get("title") or "",
                     "summary": a.get("description") or "", "source": (a.get("source") or {}).get("name", "")}
                    for a in res.get("articles", [])]
        articles += new_arts
        if new_arts:
            # persist each symbol's raw payload immediately: a crash mid-loop won't re-spend quota on retry
            pd.DataFrame(new_arts).to_csv(NEWS_RAW_CSV, mode="a",
                                          header=not os.path.exists(NEWS_RAW_CSV), index=False)
    if STAMP_NEWS:
        state["last_news_at"] = datetime.now().astimezone().isoformat(timespec="seconds")
        with open(STATE_FILE, "w") as f:
            json.dump(state, f, indent=1, sort_keys=True)
    for symbol in SYMBOLS:
        try:
            fh_news = with_retries(counted("finnhub", lambda: finnhub_client.company_news(symbol, _from=START.isoformat(),
                                                                                       to=END.isoformat())))
        except Exception as e:
            errors.append(("finnhub", symbol, str(e)[:120]))
        else:
            for n in fh_news:   # a malformed item skips itself, not the symbol's whole batch
                ts = n.get("datetime")
                if not isinstance(ts, (int, float)):
                    continue
                dt = datetime.fromtimestamp(ts, tz=timezone.utc)
                if START <= dt.date() <= END:
                    articles.append({"symbol": symbol, "date": dt.isoformat(),
                                     "headline": n.get("headline") or "", "summary": n.get("summary") or "",
                                     "source": n.get("source", "")})
        time.sleep(1.1)  # Finnhub free tier: 60 calls/min (pace successes and errors alike)
    print(f"Fetched {len(articles)} articles; {len(errors)} API errors")
    print(f"API calls: newsapi={CALLS['newsapi']} finnhub={CALLS['finnhub']}")
    if os.getenv("PIPELINE_CALLS_FILE"):                   # run_all.py adds these up for its summary
        with open(os.environ["PIPELINE_CALLS_FILE"], "a") as f:
            f.write(json.dumps(CALLS) + "\n")
    if errors:
        print(pd.DataFrame(errors, columns=["api", "symbol", "error"]).head(10).to_string(index=False))
    if MODE == "online" and not articles:
        raise RuntimeError("No articles fetched in online mode - keeping previous outputs untouched")

Fetched 9459 articles; 0 API errors
API calls: newsapi=97 finnhub=97


### Filter & score

Articles are deduped, relevance-filtered (cell above), then scored with FinBERT (`yiyanghkust/finbert-tone`).

- **Asymmetric cutoffs:** a label only counts when FinBERT is confident — ≥0.75 for positive, ≥0.65 for negative
  (negative financial language is subtler, so its bar is lower). Anything else is neutral.
- **Why neutrals are dropped from `news_cleaned_df.csv`:** that file feeds the app's AI news summaries, where neutral
  filler adds noise. But the *score* still covers every symbol — a symbol with no relevant non-neutral news gets 0,
  which the strategy reads as "no signal" rather than "missing data".

In [4]:
if MODE == "offline":
    news = pd.read_csv(NEWS_CSV)                       # already FinBERT-labelled (non-neutral only)
else:
    news = pd.DataFrame(articles, columns=["symbol", "date", "headline", "summary", "source"])
news["headline"] = news["headline"].fillna("").astype(str).str.strip()
news["summary"] = news["summary"].fillna("").astype(str).str.strip()
news = news[news["summary"] != ""].drop_duplicates(subset=["symbol", "headline", "summary"])
news = news[[is_relevant(s, f"{h} {m}") for s, h, m in zip(news["symbol"], news["headline"], news["summary"])]].copy()
news["date"] = pd.to_datetime(news["date"], utc=True, format="mixed", errors="coerce")

if MODE != "offline" and len(news):
    from transformers import pipeline
    finbert = pipeline("sentiment-analysis", model="yiyanghkust/finbert-tone", framework="pt")
    texts = [f"{h}. {m}".strip() for h, m in zip(news["headline"], news["summary"])]
    results = finbert(texts, truncation=True, max_length=512, batch_size=16)
    # Keep a label only when FinBERT clears its confidence bar (0.75 for positive, 0.65 for negative);
    # anything below the bar becomes neutral and is dropped from news_cleaned_df.csv below.
    def keep_label(result):
        label = result["label"].lower()
        bar = POSITIVE_CUTOFF if label == "positive" else NEGATIVE_CUTOFF
        return label if label in {"positive", "negative"} and result["score"] >= bar else "neutral"

    news["sentiment_label"] = [keep_label(r) for r in results]
elif MODE != "offline":
    news["sentiment_label"] = pd.Series(dtype=str)
print(f"Relevant articles: {len(news)}")
print(news["sentiment_label"].value_counts().to_string())

Device set to use mps:0


Relevant articles: 4529
sentiment_label
neutral     1926
positive    1849
negative     754


### Weighted sentiment & save

Negative articles count double. Score = (positive − 2×negative) / (positive + 2×negative + 5) × 10, so stocks with few articles stay
near 0; stocks without any relevant non-neutral article get exactly 0.

In [5]:
counts = (news.pivot_table(index="symbol", columns="sentiment_label", aggfunc="size")
          .reindex(index=SYMBOLS, columns=["positive", "negative"]).fillna(0))
pos, neg = counts["positive"], counts["negative"]
# Negative articles count double; the +5 prior pulls thinly-covered symbols toward 0.
# Score = (pos - 2*neg) / (pos + 2*neg + 5) * 10  ->  -10..+10; no relevant news -> exactly 0.
score = ((pos - 2 * neg) / (pos + 2 * neg + PRIOR_ARTICLES) * 10).round(2)
weighted = pd.DataFrame({"Symbol": counts.index, "SentimentScore": score.values,
                         "Positive": pos.astype(int).values, "Negative": neg.astype(int).values})
kept = news.loc[news["sentiment_label"] != "neutral", ["symbol", "date", "headline", "summary", "source", "sentiment_label"]]

if MODE == "sample":
    kept.to_csv(os.path.join(CACHE_DIR, "sample_news.csv"), index=False)
    weighted.to_csv(os.path.join(CACHE_DIR, "sample_weighted_sentiment.csv"), index=False)
    print("Sample mode: wrote Reports/cache/sample_*.csv only (production outputs untouched)")
elif len(news) == 0 and os.path.exists(NEWS_CSV):
    # the relevance filter emptied everything (e.g. a bad alias change): keep the previous
    # outputs untouched instead of writing an empty file and zeroing every symbol's score.
    print("WARNING: relevance filter removed every article - keeping previous news_cleaned_df.csv "
          "and weighted_sentiment.csv untouched")
else:
    kept.to_csv(NEWS_CSV, index=False)
    weighted.to_csv(SCORE_CSV, index=False)
    if MODE == "online":
        hist = weighted.assign(as_of=str(END))
        if os.path.exists(HISTORY_CSV):
            hist = pd.concat([pd.read_csv(HISTORY_CSV), hist]).drop_duplicates(["as_of", "Symbol"], keep="last")
        hist.to_csv(HISTORY_CSV, index=False)
    print(f"✅ Saved sentiment for {len(weighted)} symbols ({(weighted['SentimentScore'] != 0).sum()} non-zero), {len(kept)} articles")
weighted.sort_values("SentimentScore", ascending=False).head(10)

✅ Saved sentiment for 97 symbols (91 non-zero), 2603 articles


,Symbol,SentimentScore,Positive,Negative
19,FTNT,7.37,14,0
44,TEAM,7.22,13,0
77,TEM,7.06,12,0
41,SLB,6.88,11,0
80,TRGP,6.88,11,0
4,ANET,6.79,21,1
29,MRK,6.58,29,2
93,LITE,6.49,28,2
89,FCX,6.43,9,0
27,LRCX,5.71,14,1
